In [6]:

import re
from pathlib import Path

from lxml import etree


class PreProcessingXmi:
    """
    Deterministic preprocessing and strict validation of XMI files.

    Processing sequence:
        1. Remove ownedComment elements.
        2. Repair malformed specification body attributes.
        3. Escape invalid ampersands.
        4. Escape unescaped comparison operators in body elements.
        5. Strictly parse the XML.
        6. Write and revalidate the processed XMI.

    Original source files are never modified.
    """

    _OWNED_COMMENT_OPEN = re.compile(
        r'<(?:[\w.-]+:)?ownedComment\b'
    )

    _OWNED_COMMENT_CLOSE = re.compile(
        r'</(?:[\w.-]+:)?ownedComment\s*>'
    )

    _SPECIFICATION_BODY = re.compile(
        r'(<(?:[\w.-]+:)?specification\b[^>]*?\bbody=")'
        r'(.*?)'
        r'("(?=\s+language\s*=))',
        flags=re.DOTALL
    )

    _BODY_ELEMENT = re.compile(
        r'(<(?:[\w.-]+:)?body\b[^>]*>)'
        r'(.*?)'
        r'(</(?:[\w.-]+:)?body\s*>)',
        flags=re.DOTALL
    )

    _INVALID_AMPERSAND = re.compile(
        r'&(?!#\d+;|#x[0-9a-fA-F]+;|amp;|lt;|gt;|quot;|apos;)'
    )

    def __init__(self, input_path=None, output_path=None):
        self.input_path = Path(input_path) if input_path else None
        self.output_path = Path(output_path) if output_path else None

        self.parser = etree.XMLParser(
            recover=False,
            resolve_entities=False,
            no_network=True,
            load_dtd=False,
            remove_blank_text=False,
            strip_cdata=False,
            huge_tree=True
        )

        self.last_report = {}

    # ---------------------------------------------------------
    # 1. Remove ownedComment elements
    # ---------------------------------------------------------

    @staticmethod
    def _find_tag_end(text, start):
        """
        Find the end of an XML opening tag while respecting
        quoted attribute values.
        """
        quote = None

        for position in range(start, len(text)):
            char = text[position]

            if quote:
                if char == quote:
                    quote = None
            elif char in ('"', "'"):
                quote = char
            elif char == '>':
                return position + 1

        raise ValueError(
            f"Unterminated XML opening tag at offset {start}."
        )

    def _remove_owned_comments(self, text):
        """
        Remove self-closing and paired ownedComment elements.

        This deliberately does not parse the XML document.
        Malformed comment attributes and embedded HTML are
        therefore safe to remove before XML parsing.
        """
        removals = []
        position = 0

        while True:
            match = self._OWNED_COMMENT_OPEN.search(text, position)

            if match is None:
                break

            start = match.start()
            opening_end = self._find_tag_end(text, start)

            opening_tag = text[start:opening_end].rstrip()

            if opening_tag.endswith('/>'):
                end = opening_end
            else:
                closing_match = self._OWNED_COMMENT_CLOSE.search(
                    text, opening_end
                )

                if closing_match is None:
                    raise ValueError(
                        f"Unmatched ownedComment at offset {start}."
                    )

                end = closing_match.end()

            removals.append((start, end))
            position = end

        # Remove from right to left so earlier offsets remain valid.
        for start, end in reversed(removals):
            text = text[:start] + text[end:]

        remaining = len(self._OWNED_COMMENT_OPEN.findall(text))

        if remaining:
            raise ValueError(
                f"{remaining} ownedComment elements remain."
            )

        return text, len(removals)

    # ---------------------------------------------------------
    # 2. Repair malformed specification body attributes
    # ---------------------------------------------------------

    def _repair_body_attributes(self, text):
        """
        Escape XML-special characters inside specification body
        attributes, including raw HTML tags and quotation marks.
        """

        def repair(match):
            body = match.group(2)

            body = body.replace('&', '&amp;')
            body = body.replace('<', '&lt;')
            body = body.replace('"', '&quot;')

            # Restore valid XML entities that were already present.
            body = re.sub(
                r'&amp;(#\d+;|#x[0-9a-fA-F]+;|amp;|lt;|gt;|quot;|apos;)',
                r'&\1',
                body
            )

            return match.group(1) + body + match.group(3)

        return self._SPECIFICATION_BODY.sub(repair, text)

    # ---------------------------------------------------------
    # 3. Repair invalid ampersands
    # ---------------------------------------------------------

    def _repair_ampersands(self, text):
        """
        Escape ampersands that are not part of valid XML entities.
        """
        return self._INVALID_AMPERSAND.sub('&amp;', text)

    # ---------------------------------------------------------
    # 4. Repair OCL comparison operators in body elements
    # ---------------------------------------------------------

    def _repair_body_operators(self, text):
        """
        Escape invalid less-than operators in body element text.

        Examples:
            <= becomes &lt;=
            <> becomes &lt;&gt;
            <  becomes &lt;

        Greater-than operators do not require escaping in XML
        character data.
        """

        def repair(match):
            body = match.group(2)

            body = body.replace('<=', '&lt;=')
            body = body.replace('<>', '&lt;&gt;')

            # Handle standalone less-than comparisons.
            body = re.sub(
                r'(?<=\s)<(?=\s)',
                '&lt;',
                body
            )

            return match.group(1) + body + match.group(3)

        return self._BODY_ELEMENT.sub(repair, text)

    # ---------------------------------------------------------
    # 5. Strict XML parsing
    # ---------------------------------------------------------

    def _parse_xml(self, text, source=None):
        """
        Parse XML without recovery and provide useful error context.
        """
        try:
            return etree.fromstring(
                text.encode('utf-8'),
                self.parser
            )

        except etree.XMLSyntaxError as error:
            line_number, column = error.position
            lines = text.splitlines()

            print("\nXML PARSING FAILED")
            print(f"Source: {source}")
            print(f"Error: {error}")
            print(f"Line: {line_number}")
            print(f"Column: {column}")

            if 0 < line_number <= len(lines):
                line = lines[line_number - 1]

                print("\nProblematic line:")
                print(line)

                start = max(0, column - 100)
                end = min(len(line), column + 100)

                print("\nContext:")
                print(line[start:end])
                print(" " * (column - start) + "^")

            raise

    # ---------------------------------------------------------
    # 6. Process one XMI file
    # ---------------------------------------------------------

    def process_file(self, input_path=None, output_path=None):
        source = Path(input_path) if input_path else self.input_path
        destination = (
            Path(output_path) if output_path else self.output_path
        )

        if source is None or destination is None:
            raise ValueError(
                "Both input_path and output_path must be specified."
            )

        if not source.is_file():
            raise FileNotFoundError(source)

        raw_text = source.read_text(encoding='utf-8')

        # Execute preprocessing in the validated order.
        cleaned_text, comments_removed = (
            self._remove_owned_comments(raw_text)
        )

        cleaned_text = self._repair_body_attributes(cleaned_text)
        cleaned_text = self._repair_ampersands(cleaned_text)
        cleaned_text = self._repair_body_operators(cleaned_text)

        # Strictly parse before writing.
        root = self._parse_xml(cleaned_text, source)

        tree = etree.ElementTree(root)

        destination.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        tree.write(
            str(destination),
            encoding='UTF-8',
            xml_declaration=True,
            pretty_print=False
        )

        # Validate the actual serialized output.
        validated_tree = etree.parse(
            str(destination),
            self.parser
        )

        root = validated_tree.getroot()

        report = {
            'source': str(source),
            'output': str(destination),
            'comments_removed': comments_removed,
            'remaining_comments': len(
                root.xpath(
                    '//*[local-name()="ownedComment"]'
                )
            ),
            'elements': sum(1 for _ in root.iter()),
            'valid': True
        }

        self.last_report = report

        print(
            f"Processed: {source.name} | "
            f"Comments removed: {comments_removed} | "
            f"Elements: {report['elements']} | "
            f"XML valid: {report['valid']}"
        )

        return root, report

    # ---------------------------------------------------------
    # 7. Process an entire directory
    # ---------------------------------------------------------

    def process_directory(
        self,
        input_directory=None,
        output_directory=None,
        pattern='*.xmi',
        recursive=False
    ):
        """
        Process all matching XMI files in a directory.

        Returns:
            roots: Dictionary mapping filenames to parsed XML roots.
            reports: List of per-file processing reports.
            errors: Dictionary mapping filenames to exceptions.
        """
        source_dir = (
            Path(input_directory)
            if input_directory
            else self.input_path
        )

        destination_dir = (
            Path(output_directory)
            if output_directory
            else self.output_path
        )

        if source_dir is None or destination_dir is None:
            raise ValueError(
                "Input and output directories must be specified."
            )

        if not source_dir.is_dir():
            raise NotADirectoryError(source_dir)

        files = (
            sorted(source_dir.rglob(pattern))
            if recursive
            else sorted(source_dir.glob(pattern))
        )

        roots = {}
        reports = []
        errors = {}

        for source in files:
            relative_path = source.relative_to(source_dir)
            destination = destination_dir / relative_path

            try:
                root, report = self.process_file(
                    source,
                    destination
                )

                roots[source.name] = root
                reports.append(report)

            except Exception as error:
                errors[source.name] = error
                print(
                    f"FAILED: {source.name} | {error}"
                )

        self.last_report = {
            'files_found': len(files),
            'files_processed': len(reports),
            'files_failed': len(errors),
            'reports': reports,
            'errors': errors
        }

        print("\nBATCH PROCESSING SUMMARY")
        print(f"Files found:      {len(files)}")
        print(f"Files processed:  {len(reports)}")
        print(f"Files failed:     {len(errors)}")

        return roots, reports, errors


In [7]:
parser = PreProcessingXmi()

roots, reports, errors = parser.process_directory(
    input_directory="C:/Users/Drew/Documents/MyFiles/repo/sysml/lang_files/raw",
    output_directory="C:/Users/Drew/Documents/MyFiles/repo/sysml/lang_files/processed",
    pattern="*.xmi",
    recursive=False
)

Processed: kerml.xmi | Comments removed: 718 | Elements: 4007 | XML valid: True
Processed: sysml_1.xmi | Comments removed: 79 | Elements: 1284 | XML valid: True
Processed: sysml_1_to_2.xmi | Comments removed: 587 | Elements: 26641 | XML valid: True
Processed: sysml_2.xmi | Comments removed: 892 | Elements: 4517 | XML valid: True
Processed: uml_diagraminterchange.xmi | Comments removed: 108 | Elements: 446 | XML valid: True

XML PARSING FAILED
Source: C:\Users\Drew\Documents\MyFiles\repo\sysml\lang_files\raw\uml_metamodel.xmi
Error: Opening and ending tag mismatch: ownedOperation line 5718 and ownedRule, line 5724, column 13 (<string>, line 5724)
Line: 5724
Column: 13

Problematic line:
</ownedRule>

Context:
</ownedRule>
             ^
FAILED: uml_metamodel.xmi | Opening and ending tag mismatch: ownedOperation line 5718 and ownedRule, line 5724, column 13 (<string>, line 5724)
Processed: uml_primativetypes.xmi | Comments removed: 5 | Elements: 13 | XML valid: True
Processed: uml_standa